# Generating NIX-alone input

This script generates nix-alone input (snowpack state file and meteorological forcing) from a specified ICON run, for a specified grid point.

Modify the settings block below as needed.

### --- Settings ---

In [ ]:
# Provide base dir for the ICON output, containing lff* and iff* grib files
base_dir = "/path/to/run/"
# Provide netcdf with ICON grid description
gdf = "/path/to/icon_grid.nc"

# File to be used to derive the NIX state (typically first time step)
snowcover_state = "iff2024041001"
# File pattern to cover the meteorological forcing period
meteorological_forcing = "lff202404*"

# Requested tile
tileIndex = 2   # Typically tileIndex is 1, 2 or 3
tileAttr = 2    # Note: tileAttr = 2 denotes the snow tiles

# Requested longitude, latitude
lon=9.81
lat=46.83

# Output file names
statefile="nix.state"
forcingfile="nix.inp"

### --- End of settings ---

In [ ]:
import dask
import dask.array as da
from dask.distributed import Client, LocalCluster
import xarray as xr
import numpy as np

from icon_timeseries.field import get_grid

def select_point(grid, longitude: float, latitude: float) -> int:
    lons = grid.cx
    lats = grid.cy
    dist_squared = (lons - longitude) ** 2 + (lats - latitude) ** 2
    index = np.argmin(dist_squared)

    lat_lon_pairs = list(zip(lats, lons))

# Determine the required index that matches the requested lon, lat
selindex = select_point(get_grid(gdf), lon, lat)

### Define the output functions

In [ ]:
# Write the snow cover state to file
def write_statefile(statefile):
    with open(statefile, 'w') as file:
        file.write("validtime="+str(ds_sel.time.values)[:19]+"\n")
        file.write("albedo="+str(ds_sel['ALB_RAD'].item())+"\n")
        file.write("z0="+str(ds_sel['Z0'].item())+"\n")
        file.write("nLayers="+str(len(ds_sel.snowLayer))+"\n")
        for i in range(len(ds_sel.snowLayer)):
            if i == 0:
                hdr_line=['index']
            out_line = [int(ds_sel.snowLayer[i].item())]  # First column contains layer index
            for var in snow_vars:
                if 'snowLayer' in ds_sel[var].dims:
                    if i == 0:
                        hdr_line.append(var)
                    out_line.append(ds_sel[var].compute().isel(snowLayer=i).item())
    
            if i == 0:
                file.write(' '.join(map(str, hdr_line)) + '\n')
            file.write(' '.join(map(str, out_line)) + '\n')
    
    
        file.write("nNodes="+str(len(ds_sel.snow))+"\n")
        for i in range(len(ds_sel.snow)):
            if i == 0:
                hdr_line=['index']
            out_line = [int(ds_sel.snow[i].item())]  # First column contains layer index
            for var in snow_vars:
                if 'snow' in ds_sel[var].dims:
                    if i == 0:
                        hdr_line.append(var)
                    out_line.append(ds_sel[var].compute().isel(snow=i).item())
    
            if i == 0:
                file.write(' '.join(map(str, hdr_line)) + '\n')
            file.write(' '.join(map(str, out_line)) + '\n')


def write_forcingfile(forcingfile):
    with open(forcingfile, 'w') as file:
        for i in range(len(ds_sel.time)):
            if i == 0:
                hdr_line=['time']
            out_line = [np.datetime_as_string(ds_sel.time[i].values, unit='s')]  # First column contains layer index
            for var in param_list:
                if 'time' in ds_sel[var].dims:
                    if i == 0:
                        hdr_line.append(var)
                    out_line.append(f"{ds_sel[var].compute().isel(time=i).item():.6g}")
    
            if i == 0:
                file.write(' '.join(map(str, hdr_line)) + '\n')
            file.write(' '.join(map(str, out_line)) + '\n')

### Snow cover

Now extract the snow cover information.

In [ ]:
# List the variables to be extracted
snow_vars = ['DZ_SNOW_M', 'TofSNW_LTOP_M', 'TofSNW_NODE_M', 'AIRinSNW_VC_T_M', 'H2OinSNW_VC_T_M', 'ICEinSNW_VC_T_M']
surface_vars = ['ALB_RAD', 'Z0']

# Set up a dask array to read the variables from the file
tmp_ds_lst = [
    dask.delayed(xr.open_dataset)(
        base_dir + "/" + snowcover_state,
        engine='cfgrib',
        indexpath='',
        backend_kwargs={'errors': 'ignore', 
                        'read_keys': ['tileIndex', 'tileAttribute'],
                        'filter_by_keys': {'shortName': var,
                                           'tileIndex': tileIndex,
                                           'tileAttribute': tileAttr}
                       }
    )
    for var in snow_vars + surface_vars
]

# Merge all the dask tasks to create a dataset containing all parameters
ds = xr.merge(dask.compute(*tmp_ds_lst))
# Select the requested grid point
ds_sel = ds.isel(values=selindex).compute()

# Rename some parameters
ds_sel = ds_sel.rename({'sde': 'DZ_SNOW_M'})
ds_sel = ds_sel.rename({'tsn': 'TofSNW_LTOP_M'})
ds_sel = ds_sel.rename({'al': 'ALB_RAD'})
ds_sel = ds_sel.rename({'surface': 'Z0'})

# Write the output
write_statefile(statefile)

### Atmospheric forcing data

Now extract the time series information with the atmospheric forcing.

In [ ]:
def preprocess(ds):
    '''Preselect the grid point and variables of interest'''
    #return ds.set_coords(('time')).isel(values=selindex)
    return ds[param_list].isel(values=selindex)

param_list = [
               'T',         # Temperature at atmospheric levels       [K]
               'PS',        # Pressure                                [Pa]
               'QV',        # Specific humidity at atmospheric levels [kg/kg]
               'U',         # Wind U-component at atmospheric levels  [m/s]
               'V',         # Wind V-component at atmospheric levels  [m/s]
               'ASWDIR_S',  # Direct downwelling shortwave            [W m-2]
               'ASWDIFD_S', # Diffuse downwelling shortwave           [W m-2]
               'ATHD_S',    # Downwelling longwave                    [W m-2]
               'TOT_PREC',  # Total precipitation                     [kg m-2 s-1]
               'RAIN_GSP',  # Large scale rain rate                   [kg m-2 s-1]
               'SNOW_GSP',  # Large scale snow rate                   [kg m-2 s-1]
               'GRAU_GSP',  # Graupel precipitation rate              [kg m-2 s-1]
               'T_SO'       # Soil temperature                        [K]
             ]

print("Collecting workload")
tmp_ds_lst = []
ds_full=xr.open_mfdataset(base_dir + "/" + meteorological_forcing,
                          engine='cfgrib',
                          parallel=True,
                          concat_dim='time',
                          combine='nested',
                          preprocess=preprocess,
                          data_vars='minimal',
                          coords='minimal',
                          compat='override',
                          indexpath='',
                          backend_kwargs={'errors': 'ignore',
                                          'encode_cf': ('time', 'geography', 'vertical')
                                         }
                         )

# Surface parameters
tmp_ds_lst.append(ds_full[['PS', 'ASWDIR_S', 'ASWDIFD_S', 'ATHD_S', 'TOT_PREC', 'RAIN_GSP', 'SNOW_GSP', 'GRAU_GSP']])
# Soil parameter at highest soil layer
tmp_ds_lst.append(ds_full['T_SO'].sel(depthBelowLand=ds_full['depthBelowLand'].min().item()))
# Parameters at lowest atmospheric level
tmp_ds_lst.append(ds_full[['T', 'QV', 'U', 'V']].sel(generalVerticalLayer=ds_full['generalVerticalLayer'].max().item()))

# Merge collected data
ds=xr.merge(tmp_ds_lst, compat='override')

# Force dask to compute
print("Computing")
ds_sel = ds.compute()

# Some checks and corrections
for var in ['TOT_PREC', 'RAIN_GSP', 'SNOW_GSP', 'GRAU_GSP']:
    if (ds_full[var].attrs['GRIB_units'] != 'kg m-2' ):
        raise Exception("ERROR: expected units for " + var + " are kg m-2, found: " + ds_full[var].attrs['GRIB_units'])

# Determine the interval in seconds in the dataset
ds_interval = np.median(np.diff(ds['time'].values) / np.timedelta64(1, 's'))

# Scale the variables accordingly
# Convert precipitation rates to kg / m / s:
ds_sel['TOT_PREC'] /= ds_interval
ds_sel['RAIN_GSP'] /= ds_interval
ds_sel['SNOW_GSP'] /= ds_interval
ds_sel['GRAU_GSP'] /= ds_interval

print("Writing output")
write_forcingfile(forcingfile)